# process TradeData from un comtrade database

In [ ]:
import pandas as pd
import os

# Define input and output paths in the Downloads folder
# Using expanduser('~') finds the current user's home directory (Windows/Mac/Linux)
input_file = os.path.expanduser('~/Downloads/TradeData.xlsx')
output_file = os.path.expanduser('~/Downloads/TradeData_Processed.xlsx')

# List of Sub-Saharan Africa ISO3 codes
subsaharan_iso3 = [
    'AGO', 'BEN', 'BWA', 'BFA', 'BDI', 'CPV', 'CMR', 'CAF', 'TCD', 'COM',
    'COD', 'COG', 'CIV', 'DJI', 'GNQ', 'ERI', 'SWZ', 'ETH', 'GAB', 'GMB',
    'GHA', 'GIN', 'GNB', 'KEN', 'LSO', 'LBR', 'MDG', 'MWI', 'MLI', 'MRT',
    'MUS', 'MOZ', 'NAM', 'NER', 'NGA', 'RWA', 'STP', 'SEN', 'SYC', 'SLE',
    'SOM', 'ZAF', 'SSD', 'SDN', 'TZA', 'TGO', 'UGA', 'ZMB', 'ZWE'
]

# 1. Load the file
df = pd.read_excel(input_file)

# 2. Delete rows where the 'qty' column is < 1000
# We keep only rows where qty is greater than or equal to 1000
df = df[df['qty'] >= 1000]

# 3. Filter to keep only Sub-Saharan partner countries
df = df[df['partnerISO'].isin(subsaharan_iso3)]

# 4. Merge rows and create columns for each year ('period')
# The values populated in the new year columns will be the 'qty' amounts
df_pivot = df.pivot_table(
    index=['reporterISO', 'partnerISO'], 
    columns='period', 
    values='qty', 
    aggfunc='sum'
).reset_index()

# 5. Remove the column index name created by the pivot operation for cleaner output
df_pivot.columns.name = None

# Save the resulting dataframe to a new Excel file
df_pivot.to_excel(output_file, index=False)

print(f"Processing complete. File saved to: {output_file}")

# create a fake fob_per_kg_fake.tif

In [1]:
import rasterio
import numpy as np
from pathlib import Path

# Definisci i percorsi
data_dir = Path("dataset_first_step")
reference_raster_path = data_dir / "friction_moto.tif" # Usiamo questo come template geometrico
output_raster_path = data_dir / "fob_per_kg_fake.tif"

# Valore costante desiderato
fake_value = 0.747694

if not reference_raster_path.exists():
    raise FileNotFoundError(f"Impossibile trovare il raster di riferimento: {reference_raster_path}")

# Apri il raster di riferimento per leggerne i metadati (profilo)
with rasterio.open(reference_raster_path) as src:
    profile = src.profile
    
    # Crea un array numpy della stessa forma, riempito con il valore 0.74
    # src.shape restituisce (altezza, larghezza)
    fake_data = np.full((1, src.height, src.width), fake_value, dtype=np.float32)
    
    # Aggiorna il profilo per riflettere il tipo di dato e il numero di bande (1)
    profile.update(
        dtype=rasterio.float32,
        count=1,
        nodata=np.nan # Definisce NaN come valore nullo se necessario in futuro
    )

    # Scrivi il nuovo file
    with rasterio.open(output_raster_path, 'w', **profile) as dst:
        dst.write(fake_data)

print(f"File raster creato con successo: {output_raster_path}")
print(f"Valore costante applicato: {fake_value}")

File raster creato con successo: dataset_first_step\fob_per_kg_fake.tif
Valore costante applicato: 0.747694


# alternative processing for tradedata fob_per_kg

In [5]:
import pandas as pd


file_path = r'C:/Users/Fra/Downloads/TradeData_processed.xlsx'
df = pd.read_excel(file_path)

aggregated = df.groupby(['partnerISO', 'period'])[['fob', 'kg']].sum().reset_index()

aggregated['fob_per_kg'] = aggregated['fob'] / aggregated['kg']

final_output = aggregated.pivot(index='partnerISO', columns='period', values='fob_per_kg')

output_file = 'fob_per_kg_matrix.xlsx'
final_output.to_excel(output_file)

print(f"Completed. saved as: {output_file}")

Completed. saved as: fob_per_kg_matrix.xlsx
